
# ARC-v0.19 — Cross-Approximation FEVER Replication
## Search-Effort Approximation under Feedback (E5, IVF-SQ8, nprobe 8 vs 64)

**Purpose.** Test whether the approximation-feedback phenomenon extends beyond the earlier **representation-fidelity** contrast (IVF-PQ32 vs IVF-SQ8) to a different approximation mechanism: **search-effort approximation within the same index representation**.

ARC-v0.19 reuses the completed E5 FEVER artifacts from ARC-v0.18 and changes only IVF search effort:

- **Low fidelity:** IVF-SQ8 with `nprobe=8`
- **Higher fidelity comparator:** the *same* IVF-SQ8 index with `nprobe=64`

This isolates search approximation more cleanly than PQ-vs-SQ comparison because both trajectories use:

- the same encoder (`intfloat/e5-small-v2`),
- the same 384-d normalized embeddings,
- the same IVF-SQ8 representation,
- the same coarse centroids,
- the same corpus,
- the same query split,
- the same feedback implementation,
- the same 44 policy configurations.

Only `nprobe` differs.

The higher-fidelity condition remains a **relative comparator**, not an exact-search oracle.

---

## Frozen research questions

1. Does increasing search effort (`nprobe=8 → 64`) improve one-shot retrieval quality?
2. Do coupled low/high-search-effort trajectories produce positive aggregate H1/H2/H3 divergence?
3. Does amplification remain a minority regime at the primary threshold \(\epsilon=0.002\)?
4. Does amplification incidence increase from \(\alpha=0.1\) to \(\alpha=0.7\)?
5. Does configuration-level amplification risk reproduce from FIT to untouched validation?
6. Conditional on amplification, is signed utility divergence predominantly harmful to the lower-search-effort trajectory?

---

## Primary endpoints

For coupled trajectories initialized from the same query:

\[
H1=\operatorname{slope}\bigl(d(q_t^{L},q_t^{H})\bigr)
\]

\[
H2=\operatorname{slope}\bigl(d(R_t^{L},R_t^{H})\bigr)
\]

\[
H3_{\mathrm{abs}}
=
\operatorname{slope}
\left(
|u_H(t)-u_L(t)|
\right).
\]

Signed utility is

\[
G_t=u_H(t)-u_L(t).
\]

Thus \(G_T>0\) means the higher-search-effort trajectory ends with higher nDCG@10 and the divergence is directionally harmful to the low-`nprobe` trajectory.

---

## Frozen claim gate

Seven criteria are evaluated on untouched validation:

1. `nprobe=64` one-shot nDCG@10 \(>\) `nprobe=8`;
2. mean H1 \(>0\);
3. mean H2 \(>0\);
4. mean H3_abs \(>0\);
5. amplification fraction at \(\epsilon=0.002\) \(<0.5\);
6. amplification incidence at \(\alpha=0.7\) \(>\alpha=0.1\);
7. FIT→validation configuration-risk Pearson \(r>0.5\).

Classification:

- **CROSS_APPROX_REPLICATION**: all 7 criteria pass;
- **CROSS_APPROX_PARTIAL_REPLICATION**: 4–6 pass;
- **CROSS_APPROX_NON_REPLICATION**: 0–3 pass.

Negative and partial results are retained. Do not change `nprobe`, thresholds, encoder, or policy grid after observing outcomes.

The signed-direction result is a preregistered **secondary** endpoint and does not alter the primary cross-approximation claim gate.


In [ ]:

# ============================================================
# Cell 1 — Install dependencies
# ============================================================

%pip install -q faiss-cpu==1.12.0 pyarrow pandas numpy scipy tqdm


In [ ]:

# ============================================================
# Cell 2 — Imports / Drive / frozen constants
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import gc
import hashlib
import json
import math
import os
import random
import time
import warnings

import faiss
import numpy as np
import pandas as pd

from scipy.stats import pearsonr, spearmanr
from tqdm.auto import tqdm
from google.colab import drive

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 20260819
random.seed(SEED)
np.random.seed(SEED)

# Source encoder/index representation
ENCODER_NAME = "intfloat/e5-small-v2"
DIM = 384

# Search-effort approximation axis
NPROBE_LOW = 8
NPROBE_HIGH = 64

# Frozen experiment geometry
N_DOCS = 5_416_568
N_DEV = 6_666
N_FIT = 3_350
N_VAL = 3_316
N_CONFIGS = 44

TOP_RETRIEVE = 100
TOP_K = 10
MAX_ROUNDS = 4

EPS_PRIMARY = 0.002
EPS_SWEEP = [0.0, 0.001, 0.002, 0.005, 0.01]

# Batch execution
SEARCH_BATCH = 512
FEEDBACK_BATCH = 128

# Statistical audit
BOOTSTRAP_REPS = 5_000

DRIVE_ROOT = Path("/content/drive/MyDrive")

if not DRIVE_ROOT.is_dir():
    drive.mount("/content/drive")

assert DRIVE_ROOT.is_dir(), "Google Drive mount failed."

ARC_ROOT = DRIVE_ROOT / "rag-pq-checkpoints" / "arc-v0"

# Use all available CPU threads for faiss-cpu.
cpu_count = os.cpu_count() or 1
faiss.omp_set_num_threads(cpu_count)

print("Drive:", DRIVE_ROOT)
print("Encoder:", ENCODER_NAME)
print("nprobe:", NPROBE_LOW, "vs", NPROBE_HIGH)
print("CPU threads:", faiss.omp_get_max_threads())


In [ ]:

# ============================================================
# Cell 3 — Resolve completed ARC-v0.18 source run
# ============================================================

SOURCE_ROOT = (
    ARC_ROOT
    / "cross-encoder-fever-replication-v018"
)

preferred = SOURCE_ROOT / "20260819-015645"

def is_complete_v018(p):
    return (
        p.is_dir()
        and (p / "v018_cross_encoder_replication_report.json").is_file()
        and (p / "v018_validation_endpoints.parquet").is_file()
        and (p / "v018_fit_endpoints.parquet").is_file()
        and len(list((p / "fit").glob("fit-*.parquet"))) == N_CONFIGS
        and len(list((p / "validation").glob("validation-*.parquet"))) == N_CONFIGS
    )

if is_complete_v018(preferred):
    SOURCE_RUN = preferred
else:
    candidates = sorted(
        [p for p in SOURCE_ROOT.iterdir() if is_complete_v018(p)],
        reverse=True,
    )
    assert candidates, "No complete ARC-v0.18 run found."
    SOURCE_RUN = candidates[0]

SOURCE_REPORT = (
    SOURCE_RUN
    / "v018_cross_encoder_replication_report.json"
)

source_report = json.loads(
    SOURCE_REPORT.read_text(encoding="utf-8")
)

assert source_report["test_accessed"] is False

def sha256_file(path, chunk_size=16 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

SOURCE_REPORT_SHA = sha256_file(SOURCE_REPORT)

print("Source run:", SOURCE_RUN)
print("Source claim gate:", source_report["claim_gate"])
print("Source report SHA:", SOURCE_REPORT_SHA)
print("ARC-v0.18 SOURCE — PASS")


In [ ]:

# ============================================================
# Cell 4 — Recover authoritative FIT / validation membership
# ============================================================

V013_ROOT = (
    ARC_ROOT
    / "fever-boundary-external-replication-v013"
)

V013_RUN = V013_ROOT / "20260817-140640"

assert V013_RUN.is_dir(), V013_RUN

fit_files_v013 = sorted(V013_RUN.glob("fit-*.parquet"))
val_files_v013 = sorted(V013_RUN.glob("validation-*.parquet"))

assert len(fit_files_v013) == 44
assert len(val_files_v013) == 44

def qset(path):
    return set(
        pd.read_parquet(
            path,
            columns=["query_id"],
        )["query_id"].astype(str)
    )

FIT_IDS = qset(fit_files_v013[0])
VAL_IDS = qset(val_files_v013[0])

assert len(FIT_IDS) == N_FIT
assert len(VAL_IDS) == N_VAL
assert not (FIT_IDS & VAL_IDS)
assert len(FIT_IDS | VAL_IDS) == N_DEV

def membership_sha(ids):
    return hashlib.sha256(
        "\n".join(sorted(ids)).encode("utf-8")
    ).hexdigest()

FIT_SHA = membership_sha(FIT_IDS)
VAL_SHA = membership_sha(VAL_IDS)

print("FIT:", len(FIT_IDS), FIT_SHA)
print("VAL:", len(VAL_IDS), VAL_SHA)
print("FROZEN MEMBERSHIP — PASS")


In [ ]:

# ============================================================
# Cell 5 — Locate immutable E5 artifacts from ARC-v0.18
# ============================================================

QUERY_EMB_PATH = SOURCE_RUN / "dev_query_embeddings.float32.npy"
QUERY_IDS_PATH = SOURCE_RUN / "dev_query_ids.txt"
CORPUS_MEMMAP_PATH = SOURCE_RUN / "corpus_embeddings.float16.memmap"

SQ8_CANDIDATES = sorted(
    SOURCE_RUN.glob("fever-e5-small-v2-ivfsq8-nlist4096.faiss")
)

assert QUERY_EMB_PATH.is_file(), QUERY_EMB_PATH
assert QUERY_IDS_PATH.is_file(), QUERY_IDS_PATH
assert CORPUS_MEMMAP_PATH.is_file(), CORPUS_MEMMAP_PATH
assert SQ8_CANDIDATES, "Missing E5 IVF-SQ8 index."

SQ8_PATH = SQ8_CANDIDATES[0]

DEV_QUERY_IDS = QUERY_IDS_PATH.read_text(
    encoding="utf-8"
).splitlines()

dev_query_embeddings = np.load(QUERY_EMB_PATH)

assert len(DEV_QUERY_IDS) == N_DEV
assert dev_query_embeddings.shape == (N_DEV, DIM)

DEV_QUERY_INDEX = {
    qid: i
    for i, qid in enumerate(DEV_QUERY_IDS)
}

expected_memmap_bytes = (
    N_DOCS
    * DIM
    * np.dtype(np.float16).itemsize
)

assert (
    CORPUS_MEMMAP_PATH.stat().st_size
    == expected_memmap_bytes
)

corpus_mm = np.memmap(
    CORPUS_MEMMAP_PATH,
    dtype=np.float16,
    mode="r",
    shape=(N_DOCS, DIM),
)

sq8 = faiss.read_index(str(SQ8_PATH))

assert sq8.ntotal == N_DOCS

print("Query embeddings:", QUERY_EMB_PATH)
print("Corpus memmap:", CORPUS_MEMMAP_PATH)
print("SQ8 index:", SQ8_PATH)
print("SQ8 ntotal:", sq8.ntotal)
print("IMMUTABLE E5 ARTIFACT LOAD — PASS")


In [ ]:

# ============================================================
# Cell 6 — Load audited FEVER DEV qrels
# ============================================================

SEALED_QRELS = (
    DRIVE_ROOT
    / "hc-rars-fever-5m-untouched-confirmation-v1"
    / "stage2"
    / "dev_qrels_rows.csv"
)

assert SEALED_QRELS.is_file(), SEALED_QRELS

qr = pd.read_csv(SEALED_QRELS)

assert {
    "query-id",
    "corpus-row",
    "score",
}.issubset(qr.columns)

QRELS = defaultdict(set)

DEV_IDS = FIT_IDS | VAL_IDS

for _, r in qr.iterrows():
    qid = str(r["query-id"])

    if (
        qid in DEV_IDS
        and float(r["score"]) > 0
    ):
        QRELS[qid].add(
            int(r["corpus-row"])
        )

missing = [
    qid
    for qid in DEV_QUERY_IDS
    if not QRELS[qid]
]

assert not missing, (
    f"Missing positive qrels for {len(missing)} queries."
)

print("DEV QRELS — PASS")


In [ ]:

# ============================================================
# Cell 7 — Seal ARC-v0.19 protocol BEFORE nprobe outcomes
# ============================================================

V019_ROOT = (
    ARC_ROOT
    / "cross-approximation-nprobe-replication-v019"
)

V019_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

RUN_ID = datetime.now(timezone.utc).strftime(
    "%Y%m%d-%H%M%S"
)

OUT = V019_ROOT / RUN_ID

OUT.mkdir(
    parents=True,
    exist_ok=False,
)

PROTOCOL = {
    "status":
        "ARC_V019_CROSS_APPROXIMATION_PROTOCOL_SEALED_BEFORE_OUTCOMES",

    "created_at_utc":
        datetime.now(timezone.utc).isoformat(),

    "source_v018_run":
        str(SOURCE_RUN),

    "source_v018_report_sha256":
        SOURCE_REPORT_SHA,

    "encoder":
        ENCODER_NAME,

    "index_representation":
        "IVF-SQ8",

    "approximation_axis":
        "IVF search effort via nprobe",

    "low_nprobe":
        NPROBE_LOW,

    "high_nprobe":
        NPROBE_HIGH,

    "fit_membership_sha256":
        FIT_SHA,

    "validation_membership_sha256":
        VAL_SHA,

    "retrieval": {
        "top_retrieve": TOP_RETRIEVE,
        "utility_k": TOP_K,
        "rounds": MAX_ROUNDS,
    },

    "policy_grid": {
        "alphas": [0.1, 0.3, 0.5, 0.7],
        "mean_k": [5, 20, 50],
        "softmax_k": [5, 20],
        "softmax_temperatures": [0.05, 0.1, 0.2, 0.5],
        "configuration_count": N_CONFIGS,
    },

    "primary_epsilon":
        EPS_PRIMARY,

    "epsilon_sensitivity":
        EPS_SWEEP,

    "primary_claim_gate_criteria": [
        "one_shot_high_ndcg_gt_low",
        "H1_positive",
        "H2_positive",
        "H3_positive",
        "amplification_minority",
        "alpha_0p7_gt_0p1",
        "config_fit_val_pearson_gt_0p5",
    ],

    "claim_gate": {
        "CROSS_APPROX_REPLICATION":
            "7 of 7 primary criteria pass",

        "CROSS_APPROX_PARTIAL_REPLICATION":
            "4 to 6 primary criteria pass",

        "CROSS_APPROX_NON_REPLICATION":
            "0 to 3 primary criteria pass",
    },

    "secondary_signed_endpoint":
        "P(G_T > 0 | H3_abs > 0.002)",

    "test_accessed":
        False,

    "test_relevance_accessed":
        False,
}

PROTOCOL_PATH = (
    OUT
    / "v019_cross_approximation_protocol.json"
)

PROTOCOL_PATH.write_text(
    json.dumps(
        PROTOCOL,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

PROTOCOL_SHA = sha256_file(
    PROTOCOL_PATH
)

(
    OUT
    / "V019_PROTOCOL_SHA256.txt"
).write_text(
    f"{PROTOCOL_SHA}  {PROTOCOL_PATH.name}\n",
    encoding="utf-8",
)

print("Output:", OUT)
print("Protocol SHA:", PROTOCOL_SHA)
print("ARC-v0.19 PROTOCOL SEALED — PASS")


In [ ]:

# ============================================================
# Cell 8 — Frozen 44-policy grid
# ============================================================

CONFIGS = []

for a in [0.1, 0.3, 0.5, 0.7]:
    for k in [5, 20, 50]:
        CONFIGS.append({
            "method": "mean",
            "alpha": a,
            "k": k,
            "temperature": None,
            "config_key":
                f"mean-k{k}-a{str(a).replace('.', 'p')}-tnone",
        })

for a in [0.1, 0.3, 0.5, 0.7]:
    for k in [5, 20]:
        for t in [0.05, 0.1, 0.2, 0.5]:
            CONFIGS.append({
                "method": "softmax",
                "alpha": a,
                "k": k,
                "temperature": t,
                "config_key":
                    (
                        f"softmax-k{k}"
                        f"-a{str(a).replace('.', 'p')}"
                        f"-t{str(t).replace('.', 'p')}"
                    ),
            })

assert len(CONFIGS) == N_CONFIGS

print("CONFIGS:", len(CONFIGS))


In [ ]:

# ============================================================
# Cell 9 — Batch retrieval / utility / feedback helpers
# ============================================================

def normalize_rows(x):
    x = np.asarray(x, dtype=np.float32)
    norms = np.linalg.norm(
        x,
        axis=1,
        keepdims=True,
    )
    return x / np.maximum(norms, 1e-12)

def search_with_nprobe(Q, nprobe, k=TOP_RETRIEVE):
    sq8.nprobe = int(nprobe)

    Q = normalize_rows(Q)

    all_scores = []
    all_ids = []

    for left in range(
        0,
        len(Q),
        SEARCH_BATCH,
    ):
        right = min(
            left + SEARCH_BATCH,
            len(Q),
        )

        scores, ids = sq8.search(
            Q[left:right],
            k,
        )

        all_scores.append(scores)
        all_ids.append(ids)

    return (
        np.vstack(all_scores),
        np.vstack(all_ids),
    )

def ndcg_batch(ids, qids):
    out = np.empty(
        len(qids),
        dtype=np.float64,
    )

    discounts = (
        1.0
        /
        np.log2(
            np.arange(2, TOP_K + 2)
        )
    )

    for i, qid in enumerate(qids):
        rel = QRELS[qid]

        gains = np.fromiter(
            (
                1.0
                if int(doc) in rel
                else 0.0
                for doc in ids[i, :TOP_K]
            ),
            dtype=np.float64,
            count=TOP_K,
        )

        dcg = float(
            (gains * discounts).sum()
        )

        m = min(
            TOP_K,
            len(rel),
        )

        idcg = float(
            discounts[:m].sum()
        )

        out[i] = (
            dcg / idcg
            if idcg > 0
            else 0.0
        )

    return out

def candidate_jaccard_batch(idsL, idsH):
    out = np.empty(
        len(idsL),
        dtype=np.float64,
    )

    for i in range(len(idsL)):
        A = set(map(int, idsL[i]))
        B = set(map(int, idsH[i]))

        out[i] = (
            1.0
            -
            len(A & B)
            /
            len(A | B)
        )

    return out

def feedback_batch(scores, ids, cfg):
    N = len(ids)

    feedback_vectors = np.empty(
        (N, DIM),
        dtype=np.float32,
    )

    k = int(cfg["k"])

    for left in range(
        0,
        N,
        FEEDBACK_BATCH,
    ):
        right = min(
            left + FEEDBACK_BATCH,
            N,
        )

        id_block = ids[
            left:right,
            :k,
        ]

        docs = np.asarray(
            corpus_mm[id_block.reshape(-1)],
            dtype=np.float32,
        ).reshape(
            right - left,
            k,
            DIM,
        )

        docs /= np.maximum(
            np.linalg.norm(
                docs,
                axis=2,
                keepdims=True,
            ),
            1e-12,
        )

        if cfg["method"] == "mean":
            fb = docs.mean(axis=1)

        else:
            score_block = scores[
                left:right,
                :k,
            ].astype(np.float64)

            z = (
                score_block
                /
                float(cfg["temperature"])
            )

            z -= z.max(
                axis=1,
                keepdims=True,
            )

            w = np.exp(z)

            w /= w.sum(
                axis=1,
                keepdims=True,
            )

            fb = (
                docs
                *
                w[:, :, None]
            ).sum(axis=1)

        feedback_vectors[
            left:right
        ] = normalize_rows(fb)

    return feedback_vectors

print("BATCH HELPERS — READY")


In [ ]:

# ============================================================
# Cell 10 — One-shot nprobe fidelity audit
# ============================================================

Q_DEV = np.stack(
    [
        dev_query_embeddings[
            DEV_QUERY_INDEX[qid]
        ]
        for qid in DEV_QUERY_IDS
    ]
).astype(np.float32)

scores_low, ids_low = search_with_nprobe(
    Q_DEV,
    NPROBE_LOW,
)

scores_high, ids_high = search_with_nprobe(
    Q_DEV,
    NPROBE_HIGH,
)

u_low = ndcg_batch(
    ids_low,
    DEV_QUERY_IDS,
)

u_high = ndcg_batch(
    ids_high,
    DEV_QUERY_IDS,
)

one_shot_summary = pd.DataFrame([
    {
        "condition":
            f"SQ8_nprobe_{NPROBE_LOW}",
        "ndcg10":
            float(u_low.mean()),
    },
    {
        "condition":
            f"SQ8_nprobe_{NPROBE_HIGH}",
        "ndcg10":
            float(u_high.mean()),
    },
])

display(one_shot_summary)

one_shot_summary.to_csv(
    OUT / "v019_one_shot_nprobe_summary.csv",
    index=False,
)

ONE_SHOT_HIGH_GT_LOW = bool(
    u_high.mean()
    >
    u_low.mean()
)

print(
    "High > Low:",
    ONE_SHOT_HIGH_GT_LOW,
)

if not ONE_SHOT_HIGH_GT_LOW:
    raise RuntimeError(
        "Frozen nprobe=64 condition does not improve DEV nDCG@10 "
        "over nprobe=8. Stop and retain this as an invalid fidelity "
        "ladder outcome; do not change nprobe after seeing results."
    )

print("ONE-SHOT NPROBE FIDELITY AUDIT — PASS")


In [ ]:

# ============================================================
# Cell 11 — Batched coupled-trajectory runner
# ============================================================

def run_config_batched(qids, cfg):
    qids = list(qids)

    q0 = np.stack(
        [
            dev_query_embeddings[
                DEV_QUERY_INDEX[qid]
            ]
            for qid in qids
        ]
    ).astype(np.float32)

    q0 = normalize_rows(q0)

    qL = q0.copy()
    qH = q0.copy()

    round_frames = []

    for t in range(
        MAX_ROUNDS + 1
    ):
        scoresL, idsL = search_with_nprobe(
            qL,
            NPROBE_LOW,
        )

        scoresH, idsH = search_with_nprobe(
            qH,
            NPROBE_HIGH,
        )

        uL = ndcg_batch(
            idsL,
            qids,
        )

        uH = ndcg_batch(
            idsH,
            qids,
        )

        state_dist = (
            1.0
            -
            np.sum(
                normalize_rows(qL)
                *
                normalize_rows(qH),
                axis=1,
            )
        )

        cand_dist = (
            candidate_jaccard_batch(
                idsL,
                idsH,
            )
        )

        signed_gap = (
            uH - uL
        )

        round_frames.append(
            pd.DataFrame({
                "query_id":
                    qids,

                "iteration":
                    t,

                "method":
                    cfg["method"],

                "alpha":
                    cfg["alpha"],

                "k":
                    cfg["k"],

                "temperature":
                    cfg["temperature"],

                "config_key":
                    cfg["config_key"],

                "query_state_distance":
                    state_dist,

                "candidate_jaccard_distance":
                    cand_dist,

                "utility_low":
                    uL,

                "utility_high":
                    uH,

                "signed_utility_gap":
                    signed_gap,

                "abs_utility_gap":
                    np.abs(
                        signed_gap
                    ),
            })
        )

        if t == MAX_ROUNDS:
            break

        fbL = feedback_batch(
            scoresL,
            idsL,
            cfg,
        )

        fbH = feedback_batch(
            scoresH,
            idsH,
            cfg,
        )

        a = float(
            cfg["alpha"]
        )

        qL = normalize_rows(
            (1.0 - a) * q0
            +
            a * fbL
        )

        qH = normalize_rows(
            (1.0 - a) * q0
            +
            a * fbH
        )

    return pd.concat(
        round_frames,
        ignore_index=True,
    )

print("BATCH TRAJECTORY RUNNER — READY")


In [ ]:

# ============================================================
# Cell 12 — Small deterministic parity / invariants smoke test
#
# We verify the batched runner reproduces itself deterministically
# and satisfies trajectory invariants before the full sweep.
# ============================================================

smoke_qids = sorted(FIT_IDS)[:16]
smoke_cfg = CONFIGS[0]

smoke_a = run_config_batched(
    smoke_qids,
    smoke_cfg,
)

smoke_b = run_config_batched(
    smoke_qids,
    smoke_cfg,
)

assert len(smoke_a) == (
    len(smoke_qids)
    * (MAX_ROUNDS + 1)
)

assert smoke_a.equals(
    smoke_b
), (
    "Batched execution is not deterministic."
)

assert np.isfinite(
    smoke_a.select_dtypes(
        include=[np.number]
    ).to_numpy()
).all()

assert (
    smoke_a["query_id"].nunique()
    == len(smoke_qids)
)

assert (
    smoke_a["iteration"].nunique()
    == MAX_ROUNDS + 1
)

display(
    smoke_a.head(10)
)

print("BATCH EXECUTION SMOKE TEST — PASS")


In [ ]:

# ============================================================
# Cell 13 — Full FIT / validation sweeps with checkpoint reuse
# ============================================================

FIT_DIR = OUT / "fit"
VAL_DIR = OUT / "validation"

FIT_DIR.mkdir(exist_ok=True)
VAL_DIR.mkdir(exist_ok=True)

def run_sweep(
    ids,
    directory,
    prefix,
):
    qids = sorted(ids)

    for j, cfg in enumerate(
        CONFIGS,
        1,
    ):
        path = (
            directory
            /
            f"{prefix}-{cfg['config_key']}.parquet"
        )

        if path.is_file():
            df = pd.read_parquet(
                path
            )

            assert (
                df["query_id"].nunique()
                == len(qids)
            )

            assert (
                len(df)
                ==
                len(qids)
                * (MAX_ROUNDS + 1)
            )

            print(
                f"[{prefix.upper()} {j:02d}/44] REUSE",
                path.name,
            )

            continue

        print()
        print("=" * 90)
        print(
            f"[{prefix.upper()} {j:02d}/44]",
            cfg["config_key"],
        )
        print("=" * 90)

        t0 = time.perf_counter()

        df = run_config_batched(
            qids,
            cfg,
        )

        assert (
            df["query_id"].nunique()
            == len(qids)
        )

        assert (
            len(df)
            ==
            len(qids)
            * (MAX_ROUNDS + 1)
        )

        df.to_parquet(
            path,
            index=False,
        )

        elapsed = (
            time.perf_counter()
            - t0
        )

        print(
            "SAVED",
            path.name,
            "seconds",
            elapsed,
            "sha256",
            sha256_file(path),
        )

run_sweep(
    FIT_IDS,
    FIT_DIR,
    "fit",
)

print("FIT SWEEP — COMPLETE")

run_sweep(
    VAL_IDS,
    VAL_DIR,
    "validation",
)

print("VALIDATION SWEEP — COMPLETE")


In [ ]:

# ============================================================
# Cell 14 — Reconstruct query-policy endpoints
# ============================================================

group_cols = [
    "query_id",
    "method",
    "alpha",
    "k",
    "temperature",
    "config_key",
]

def endpoint_df(traj):
    rows = []

    for keys, g in traj.groupby(
        group_cols,
        dropna=False,
        sort=False,
    ):
        g = g.sort_values(
            "iteration"
        )

        x = g[
            "iteration"
        ].to_numpy(
            np.float64
        )

        signed = g[
            "signed_utility_gap"
        ].to_numpy(
            np.float64
        )

        abs_gap = g[
            "abs_utility_gap"
        ].to_numpy(
            np.float64
        )

        row = dict(
            zip(
                group_cols,
                keys,
            )
        )

        row.update({
            "H1_slope":
                float(
                    np.polyfit(
                        x,
                        g[
                            "query_state_distance"
                        ],
                        1,
                    )[0]
                ),

            "H2_slope":
                float(
                    np.polyfit(
                        x,
                        g[
                            "candidate_jaccard_distance"
                        ],
                        1,
                    )[0]
                ),

            "H3_abs_slope":
                float(
                    np.polyfit(
                        x,
                        abs_gap,
                        1,
                    )[0]
                ),

            "H3_signed_slope":
                float(
                    np.polyfit(
                        x,
                        signed,
                        1,
                    )[0]
                ),

            "final_signed_gap":
                float(
                    signed[-1]
                ),
        })

        rows.append(row)

    return pd.DataFrame(rows)

def load_endpoints(
    directory,
    prefix,
):
    frames = []

    paths = sorted(
        directory.glob(
            f"{prefix}-*.parquet"
        )
    )

    assert len(paths) == N_CONFIGS

    for path in tqdm(
        paths,
        desc=f"{prefix} endpoints",
    ):
        frames.append(
            endpoint_df(
                pd.read_parquet(path)
            )
        )

    return pd.concat(
        frames,
        ignore_index=True,
    )

fit_ep = load_endpoints(
    FIT_DIR,
    "fit",
)

val_ep = load_endpoints(
    VAL_DIR,
    "validation",
)

assert (
    fit_ep["config_key"].nunique()
    == N_CONFIGS
)

assert (
    val_ep["config_key"].nunique()
    == N_CONFIGS
)

fit_ep.to_parquet(
    OUT / "v019_fit_endpoints.parquet",
    index=False,
)

val_ep.to_parquet(
    OUT / "v019_validation_endpoints.parquet",
    index=False,
)

print(
    "ENDPOINTS:",
    fit_ep.shape,
    val_ep.shape,
)


In [ ]:

# ============================================================
# Cell 15 — Aggregate H1/H2/H3
# ============================================================

aggregate = pd.DataFrame([
    {
        "split":
            "FIT",

        "H1_mean":
            fit_ep[
                "H1_slope"
            ].mean(),

        "H2_mean":
            fit_ep[
                "H2_slope"
            ].mean(),

        "H3_abs_mean":
            fit_ep[
                "H3_abs_slope"
            ].mean(),

        "H3_signed_mean":
            fit_ep[
                "H3_signed_slope"
            ].mean(),
    },
    {
        "split":
            "VALIDATION",

        "H1_mean":
            val_ep[
                "H1_slope"
            ].mean(),

        "H2_mean":
            val_ep[
                "H2_slope"
            ].mean(),

        "H3_abs_mean":
            val_ep[
                "H3_abs_slope"
            ].mean(),

        "H3_signed_mean":
            val_ep[
                "H3_signed_slope"
            ].mean(),
    },
])

display(aggregate)

aggregate.to_csv(
    OUT / "v019_aggregate_endpoints.csv",
    index=False,
)


In [ ]:

# ============================================================
# Cell 16 — Regime threshold sensitivity
# ============================================================

def regime(d, eps):
    h = d[
        "H3_abs_slope"
    ].to_numpy(
        np.float64
    )

    return {
        "epsilon":
            eps,

        "stable_or_null_fraction":
            float(
                (
                    np.abs(h)
                    <= eps
                ).mean()
            ),

        "amplifying_fraction":
            float(
                (
                    h > eps
                ).mean()
            ),

        "reversal_fraction":
            float(
                (
                    h < -eps
                ).mean()
            ),

        "exact_zero_fraction":
            float(
                (
                    h == 0
                ).mean()
            ),
    }

rows = []

for split_name, d in [
    ("FIT", fit_ep),
    ("VALIDATION", val_ep),
]:
    for eps in EPS_SWEEP:
        rows.append({
            "split":
                split_name,
            **regime(
                d,
                eps,
            ),
        })

regimes = pd.DataFrame(
    rows
)

display(regimes)

regimes.to_csv(
    OUT / "v019_regime_threshold_sensitivity.csv",
    index=False,
)


In [ ]:

# ============================================================
# Cell 17 — Alpha dose response
# ============================================================

rows = []

for split_name, d in [
    ("FIT", fit_ep),
    ("VALIDATION", val_ep),
]:
    for alpha, g in d.groupby(
        "alpha"
    ):
        rows.append({
            "split":
                split_name,

            "alpha":
                float(alpha),

            "amplifying_fraction":
                float(
                    (
                        g[
                            "H3_abs_slope"
                        ]
                        >
                        EPS_PRIMARY
                    ).mean()
                ),

            "mean_H3_abs_slope":
                float(
                    g[
                        "H3_abs_slope"
                    ].mean()
                ),
        })

dose = (
    pd.DataFrame(rows)
    .sort_values(
        [
            "split",
            "alpha",
        ]
    )
)

display(dose)

dose.to_csv(
    OUT / "v019_alpha_dose_response.csv",
    index=False,
)


In [ ]:

# ============================================================
# Cell 18 — Configuration reproducibility
# ============================================================

def cfg_risk(d):
    return (
        d.assign(
            amplifying=
                d["H3_abs_slope"]
                > EPS_PRIMARY
        )
        .groupby(
            [
                "method",
                "alpha",
                "k",
                "temperature",
                "config_key",
            ],
            dropna=False,
            as_index=False,
        )
        .agg(
            amplification_fraction=
                (
                    "amplifying",
                    "mean",
                ),

            mean_H3=
                (
                    "H3_abs_slope",
                    "mean",
                ),
        )
    )

f = cfg_risk(
    fit_ep
)

v = cfg_risk(
    val_ep
)

cfg = f.merge(
    v,
    on=[
        "method",
        "alpha",
        "k",
        "temperature",
        "config_key",
    ],
    suffixes=(
        "_fit",
        "_val",
    ),
    validate="one_to_one",
)

pearson = pearsonr(
    cfg[
        "amplification_fraction_fit"
    ],
    cfg[
        "amplification_fraction_val"
    ],
)

spearman = spearmanr(
    cfg[
        "amplification_fraction_fit"
    ],
    cfg[
        "amplification_fraction_val"
    ],
)

cfg[
    "fit_centered"
] = (
    cfg[
        "amplification_fraction_fit"
    ]
    -
    cfg.groupby(
        "alpha"
    )[
        "amplification_fraction_fit"
    ].transform(
        "mean"
    )
)

cfg[
    "val_centered"
] = (
    cfg[
        "amplification_fraction_val"
    ]
    -
    cfg.groupby(
        "alpha"
    )[
        "amplification_fraction_val"
    ].transform(
        "mean"
    )
)

pearson_centered = pearsonr(
    cfg[
        "fit_centered"
    ],
    cfg[
        "val_centered"
    ],
)

spearman_centered = spearmanr(
    cfg[
        "fit_centered"
    ],
    cfg[
        "val_centered"
    ],
)

print("Pearson:", pearson)
print("Spearman:", spearman)
print(
    "Alpha-centered Pearson:",
    pearson_centered,
)
print(
    "Alpha-centered Spearman:",
    spearman_centered,
)

cfg.to_csv(
    OUT / "v019_configuration_reproducibility.csv",
    index=False,
)


In [ ]:

# ============================================================
# Cell 19 — Preregistered secondary signed-direction endpoint
# ============================================================

amp = val_ep[
    val_ep[
        "H3_abs_slope"
    ]
    >
    EPS_PRIMARY
].copy()

TOL = 1e-12

amp[
    "is_harmful_final"
] = (
    amp[
        "final_signed_gap"
    ]
    > TOL
).astype(int)

harmful_fraction = float(
    amp[
        "is_harmful_final"
    ].mean()
)

# Query-cluster bootstrap
q = (
    amp.groupby(
        "query_id",
        as_index=False,
    )
    .agg(
        numerator=
            (
                "is_harmful_final",
                "sum",
            ),

        denominator=
            (
                "is_harmful_final",
                "size",
            ),
    )
)

numer = q[
    "numerator"
].to_numpy(
    np.float64
)

denom = q[
    "denominator"
].to_numpy(
    np.float64
)

rng = np.random.default_rng(
    SEED + 1901
)

boots = np.empty(
    BOOTSTRAP_REPS,
    dtype=np.float64,
)

for b in range(
    BOOTSTRAP_REPS
):
    idx = rng.integers(
        0,
        len(q),
        size=len(q),
    )

    boots[b] = (
        numer[idx].sum()
        /
        denom[idx].sum()
    )

signed_ci = np.quantile(
    boots,
    [0.025, 0.975],
)

signed_summary = {
    "amplification_events":
        int(len(amp)),

    "affected_queries":
        int(
            amp[
                "query_id"
            ].nunique()
        ),

    "harmful_final_fraction":
        harmful_fraction,

    "ci_low":
        float(
            signed_ci[0]
        ),

    "ci_high":
        float(
            signed_ci[1]
        ),

    "bootstrap_reps":
        BOOTSTRAP_REPS,
}

print(
    json.dumps(
        signed_summary,
        indent=2,
    )
)

(
    OUT
    / "v019_signed_direction_summary.json"
).write_text(
    json.dumps(
        signed_summary,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)


In [ ]:

# ============================================================
# Cell 20 — Frozen primary claim gate
# ============================================================

val_agg = (
    aggregate[
        aggregate[
            "split"
        ]
        == "VALIDATION"
    ]
    .iloc[0]
)

primary_regime = (
    regimes[
        (
            regimes[
                "split"
            ]
            == "VALIDATION"
        )
        &
        np.isclose(
            regimes[
                "epsilon"
            ],
            EPS_PRIMARY,
        )
    ]
    .iloc[0]
)

val_dose = (
    dose[
        dose[
            "split"
        ]
        == "VALIDATION"
    ]
    .sort_values(
        "alpha"
    )
    .reset_index(
        drop=True
    )
)

criteria = {
    "one_shot_high_ndcg_gt_low":
        ONE_SHOT_HIGH_GT_LOW,

    "H1_positive":
        bool(
            val_agg[
                "H1_mean"
            ]
            > 0
        ),

    "H2_positive":
        bool(
            val_agg[
                "H2_mean"
            ]
            > 0
        ),

    "H3_positive":
        bool(
            val_agg[
                "H3_abs_mean"
            ]
            > 0
        ),

    "amplification_minority":
        bool(
            primary_regime[
                "amplifying_fraction"
            ]
            < 0.5
        ),

    "alpha_0p7_gt_0p1":
        bool(
            val_dose.iloc[-1][
                "amplifying_fraction"
            ]
            >
            val_dose.iloc[0][
                "amplifying_fraction"
            ]
        ),

    "config_fit_val_pearson_gt_0p5":
        bool(
            pearson.statistic
            > 0.5
        ),
}

n_pass = sum(
    criteria.values()
)

if n_pass == 7:
    CLAIM_GATE = (
        "CROSS_APPROX_REPLICATION"
    )
elif n_pass >= 4:
    CLAIM_GATE = (
        "CROSS_APPROX_PARTIAL_REPLICATION"
    )
else:
    CLAIM_GATE = (
        "CROSS_APPROX_NON_REPLICATION"
    )

print("=" * 90)
print("ARC-v0.19 CROSS-APPROXIMATION CLAIM GATE")
print("=" * 90)

for key, value in criteria.items():
    print(
        f"{key:38s}",
        value,
    )

print()
print(
    "PASS:",
    n_pass,
    "/ 7",
)

print(
    "CLAIM GATE:",
    CLAIM_GATE,
)

print()
print(
    "Secondary harmful fraction:",
    harmful_fraction,
    "95% CI",
    signed_ci.tolist(),
)


In [ ]:

# ============================================================
# Cell 21 — Seal ARC-v0.19 final report
# ============================================================

report = {
    "status":
        "ARC_V019_CROSS_APPROXIMATION_REPLICATION_COMPLETE",

    "protocol_sha256":
        PROTOCOL_SHA,

    "source_v018_run":
        str(SOURCE_RUN),

    "source_v018_report_sha256":
        SOURCE_REPORT_SHA,

    "encoder":
        ENCODER_NAME,

    "index_representation":
        "IVF-SQ8",

    "approximation_axis":
        "nprobe",

    "low_nprobe":
        NPROBE_LOW,

    "high_nprobe":
        NPROBE_HIGH,

    "one_shot_summary":
        one_shot_summary.to_dict(
            "records"
        ),

    "aggregate_endpoints":
        aggregate.to_dict(
            "records"
        ),

    "primary_validation_regime":
        primary_regime.to_dict(),

    "validation_alpha_dose_response":
        val_dose.to_dict(
            "records"
        ),

    "configuration_reproducibility": {
        "pearson_r":
            float(
                pearson.statistic
            ),

        "spearman_rho":
            float(
                spearman.statistic
            ),

        "alpha_centered_pearson_r":
            float(
                pearson_centered.statistic
            ),

        "alpha_centered_spearman_rho":
            float(
                spearman_centered.statistic
            ),
    },

    "secondary_signed_direction":
        signed_summary,

    "claim_gate_criteria":
        criteria,

    "claim_gate":
        CLAIM_GATE,

    "test_accessed":
        False,

    "test_relevance_accessed":
        False,

    "interpretation_constraints": [
        (
            "ARC-v0.19 tests search-effort approximation using "
            "nprobe within one E5 IVF-SQ8 representation."
        ),
        (
            "nprobe=64 is a relative higher-search-effort "
            "comparator, not an exact-search oracle."
        ),
        (
            "The low/high nprobe values and claim gate were sealed "
            "before the nprobe outcomes were inspected."
        ),
        (
            "The signed-direction endpoint is secondary and does "
            "not alter the primary cross-approximation claim gate."
        ),
        (
            "No FEVER test outcomes are used."
        ),
    ],

    "completed_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}

REPORT_PATH = (
    OUT
    / "v019_cross_approximation_replication_report.json"
)

REPORT_PATH.write_text(
    json.dumps(
        report,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

REPORT_SHA = sha256_file(
    REPORT_PATH
)

(
    OUT
    / "V019_REPORT_SHA256.txt"
).write_text(
    f"{REPORT_SHA}  {REPORT_PATH.name}\n",
    encoding="utf-8",
)

print()
print("=" * 90)
print(
    "ARC-v0.19 CROSS-APPROXIMATION FEVER REPLICATION — COMPLETE"
)
print(
    "Claim gate:",
    CLAIM_GATE,
)
print(
    "Output:",
    OUT,
)
print(
    "Report SHA-256:",
    REPORT_SHA,
)
print(
    "Test accessed:",
    False,
)
print("=" * 90)
